# 16 — Endpoint-disjoint holdout
Chỉ chạy dataset/task qua cổng class coverage. Không diễn giải kết quả là cross-network generalization.

In [ ]:
from pathlib import Path
import json, subprocess
import pandas as pd
from IPython.display import display
ROOT = Path.cwd()
if not (ROOT / 'research/server').is_dir(): raise RuntimeError('Hãy mở Jupyter tại thư mục gốc của repo')
def stage(name): subprocess.run(['bash', 'research/server/run_next_experiments.sh', name], cwd=ROOT, check=True)

In [ ]:
stage('endpoint_prepare')

In [ ]:
prepare = json.loads((ROOT / 'research/results/endpoint_holdout_prepare.json').read_text())
assert prepare['complete'] is True
assert all(x['cross_split_endpoint_ips'] == 0 for x in prepare['datasets'].values())
display(pd.DataFrame(prepare['datasets']).T[['source_rows', 'kept_rows', 'dropped_rows', 'retained_fraction', 'cross_split_endpoint_ips']])
print('Eligible cells:', prepare['eligible_dataset_tasks'])

In [ ]:
stage('endpoint_train')

In [ ]:
stage('endpoint_verify')

In [ ]:
verified = json.loads((ROOT / 'research/results/endpoint_holdout_verification/matrix_manifest.json').read_text())
assert verified['complete'] is True
frames = []
for cell in verified['completed_dataset_tasks']:
    cell_name = f"{cell['dataset']}__{cell['task']}"
    frame = pd.read_csv(ROOT / 'research/artifacts/endpoint_holdout_runs' / cell_name / 'runs.csv')
    assert len(frame) == 15
    frames.append(frame)
endpoint_runs = pd.concat(frames, ignore_index=True)
display(endpoint_runs.groupby(['dataset', 'task', 'model']).test_macro_f1.agg(['count', 'mean', 'std', 'min', 'max']))
print(json.dumps(verified, indent=2))